## Gesture Recognition with RNNs - SmartWatch Datasets

This datasets contains eight users information about 20 gestures, with 20 reptitions which make total 3200 sequences. Each sequence: variable-length 3-axis accelerometer readings (x, y, z). The task is to classify whic 20 gestures was performed. 

In [27]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from typing import List
from torch.nn.utils.rnn import pad_sequence
from mltrainer import rnn_models, Trainer
from torch import optim
from dataclasses import dataclass
from mads_datasets import datatools
import mltrainer
mltrainer.__version__
from torch import Tensor

# 1 Iterators
We will be using an interesting dataset. [link](https://tev.fbk.eu/resources/smartwatch)

From the site:
> The SmartWatch Gestures Dataset has been collected to evaluate several gesture recognition algorithms for interacting with mobile applications using arm gestures. Eight different users performed twenty repetitions of twenty different gestures, for a total of 3200 sequences. Each sequence contains acceleration data from the 3-axis accelerometer of a first generation Sony SmartWatch™, as well as timestamps from the different clock sources available on an Android device. The smartwatch was worn on the user's right wrist. 

In [81]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor
preprocessor = PaddedPreprocessor()

#gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
#streamers = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
#train = streamers["train"]
#valid = streamers["valid"]

In [80]:
def get_gesture_streamers(batchsize: int):
    preprocessor = PaddedPreprocessor()
    factory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
    streamers = factory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
    return streamers["train"].stream(), streamers["valid"].stream()

In [ ]:
#trainstreamer = train.stream()
#validstreamer = valid.stream()
#x, y = next(iter(trainstreamer))
#x.shape, y

(torch.Size([32, 32, 3]),
 tensor([ 9, 14, 18,  2, 12, 19, 10,  6, 18,  9, 12,  8,  6,  6, 17, 19,  6, 13,
         15,  9, 19,  4,  0,  4,  5, 18, 17, 16,  0,  0,  8, 15]))

In [82]:
#print(f"Input shape : {x.shape}  → (batch, timesteps, features)")
#print(f"Label shape : {y.shape}  → (batch,)  values 0–19")
#print(f"Blind guess accuracy: {1/20:.1%}  (20 classes, uniform random)")

## Device setup

In [83]:
def get_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda:0")
    return torch.device("cpu")

# Loss and metrics

In [12]:

from mltrainer.metrics import Accuracy
from mltrainer import TrainerSettings, ReportTypes
loss_fn = torch.nn.CrossEntropyLoss()  #as cross entropy containes both logsoftmax and NLloss
accuracy = Accuracy()

## Trainer settings

In [11]:
modeldir = Path("gestures").resolve()
modeldir.mkdir(parents = True, exist_ok= True)

In [15]:
settings = TrainerSettings(
    epochs=3, # increase this to about 100 for training
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs = {
        "save": False, # save every best model, and restore the best one
        "verbose": True,
        "patience": 5, # number of epochs with no improvement after which training will be stopped
        "delta": 0.0, # minimum change to be considered an improvement
    }
)
settings


epochs: 3
metrics: [Accuracy]
logdir: gestures
train_steps: 81
valid_steps: 20
reporttypes: [<ReportTypes.TOML: 'TOML'>, <ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.MLFLOW: 'MLFLOW'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 5, 'delta': 0.0}

## Model 1 — Baseline GRU

#The simplest possible RNN classifier:
#Input → GRU → take last hidden state → Linear → 20 logits




Why GRU over vanilla RNN?
 GRU has reset and update gates that help it remember relevant information over long sequences. Vanilla RNN suffers from vanishing gradients past 10-20 timesteps. Gesture sequences can be 30-50+ steps long.

In [ ]:
#@dataclass
#class GRUConfig:
    #input_size: int   = 3    # x, y, z accelerometer
    #hidden_size: int  = 64
   # num_layers: int   = 1
   # output_size: int  = 20   # 20 gesture classes
   # dropout: float    = 0.0

In [28]:
class GRUModel(nn.Module):
    """
    Basline GRU Classifier. 
    Forward pass:
    x: (batch, timestamp, 3)
    GRU output all the hidden states: (batch, timestamps, hidden_size)
    take x[:, -1, :] only the last timestamp's hidden state
    Linear maps hidden_size → output_size (20 class logits)
"""


    def __init__(self, config: GRUConfig) -> None:
        super().__init__()
        self.gru = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            dropout=config.dropout if config.num_layers > 1 else 0.0,
            batch_first=True,   # input shape: (batch, seq, features)
        )
        self.classifier = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        out, _ = self.gru(x)          # out: (batch, seq, hidden_size)
        last = out[:, -1, :]          # last hidden: (batch, hidden_size)
        return self.classifier(last)  # logits: (batch, 20)

In [ ]:
#quick shape check 
#cfg = GRUConfig()
#model = GRUModel(cfg)

In [ ]:
#yhat = model(x)

In [ ]:
#print(f"Output shape: {yhat.shape}  → (batch=32, classes=20) ✓")
#print(f"Initial accuracy (random weights): {accuracy(y, yhat):.3f}")

Output shape: torch.Size([32, 20])  → (batch=32, classes=20) ✓
Initial accuracy (random weights): 0.031


## Model 2 — GRU with Dropout (improved regularisation)

Hypothesis: the baseline overfits. Adding dropout between GRU layers and before the output should improve generalisation.
Note: nn.GRU's `dropout` parameter only applies BETWEEN layers, not after the final layer. We add that separately.

In [34]:
class GRUWithDropout(nn.Module):
    """
    GRU with dropout after the final hidden state.
    Dropout between layers is handled by nn.GRU's built-in dropout param
    (only active when num_layers > 1).
    """
    def __init__(self, config: GRUConfig) -> None:
        super().__init__()
        self.gru = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            dropout=config.dropout if config.num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(p=config.dropout)
        self.classifier = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        out, _ = self.gru(x)
        last = out[:, -1, :]
        last = self.dropout(last)       # regularise before linear
        return self.classifier(last)
 

## Model 3 — LSTM

LSTM has an extra "cell state" in addition to the hidden state. The cell state acts as a longer-term memory, controlled by forget, input, and output gates. Sometime we prefer LSTM over GRU because of Longer sequences with longer-range dependencies. GRU has fewer parameters → faster, less data needed

In [38]:
class LSTMModel(nn.Module):
    """
    LSTM classifier — identical structure to GRUModel,
    except nn.LSTM returns (output, (h_n, c_n)) — we unpack accordingly.
    """
    def __init__(self, config: GRUConfig) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            dropout=config.dropout if config.num_layers > 1 else 0.0,
            batch_first=True,)
        self.dropout = nn.Dropout(p=config.dropout)
        self.classifier = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        out, (h_n, c_n) = self.lstm(x)  # note: LSTM returns tuple
        last = out[:, -1, :]
        last = self.dropout(last)
        return self.classifier(last)

## Model 4 — Conv1d + GRU (best architecture for gesture data)

Hypothesis: raw accelerometer signals benefit from local feature extraction. before the recurrent layer. Conv1d slides a filter along the TIME axis, similar to how Conv2d slides over spatial dims in images.

In [ ]:
#@dataclass
#class ConvGRUConfig:
  #  input_size: int    = 3
  #  conv_filters: int  = 32
  #  kernel_size: int   = 3     # look at 3 timesteps at once
  #  hidden_size: int   = 128
   # num_layers: int    = 2
  #  output_size: int   = 20
 #   dropout: float     = 0.3

In [90]:
class Conv1dGRU(nn.Module):
    """
    Conv1d feature extractor + GRU classifier.
    conv_filters and kernel_size come from the hyperopt search space.
    """
    def __init__(
        self,
        hidden_size: int,
        num_layers: int,
        conv_filters: int,
        kernel_size: int,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(3, conv_filters, kernel_size=kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(conv_filters),
            nn.ReLU(),
        )
        self.rnn = nn.GRU(
            input_size=conv_filters,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(hidden_size, 20)
 
    def forward(self, x: Tensor) -> Tensor:
        x = x.permute(0, 2, 1)        # (batch, 3, T)
        x = self.conv(x)               # (batch, conv_filters, T)
        x = x.permute(0, 2, 1)        # (batch, T, conv_filters)
        out, _ = self.rnn(x)
        return self.classifier(self.dropout(out[:, -1, :]))

In [ ]:
#conv_cfg = ConvGRUConfig()
#conv_model = Conv1dGRU(conv_cfg)
#yhat = conv_model(x)
#print(f"Conv1dGRU output shape: {yhat.shape} ✓")

Conv1dGRU output shape: torch.Size([32, 20]) ✓


# MLflow experiment runner

In [87]:
from pathlib import Path
import mlflow
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from hyperopt.pyll import scope
from loguru import logger
from mltrainer import metrics


In [99]:

def setup_mlflow(experiment_path: str) -> None:
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("gestures-hyperopt")

In [93]:
from datetime import datetime

In [88]:
def objective(params):
    modeldir = Path("models").resolve()
    if not modeldir.exists():
        modeldir.mkdir(parents=True)
        logger.info(f"Created {modeldir}")
 
    batchsize = 32
    trainstreamer, validstreamer = get_gesture_streamers(batchsize)
    accuracy = metrics.Accuracy()
 
    settings = TrainerSettings(
        epochs=20,
        metrics=[accuracy],
        logdir=Path("gestures"),
        train_steps=100,
        valid_steps=50,
        reporttypes=[ReportTypes.MLFLOW],
        scheduler_kwargs={"factor": 0.5, "patience": 3},
        earlystop_kwargs={
            "save": False,
            "verbose": False,
            "patience": 5,
            "delta": 0.001,
        },
    )
 
    device = get_device()
 
    with mlflow.start_run():
        mlflow.set_tag("model", "Conv1dGRU")
        mlflow.set_tag("dev", "your-name")
 
        # Log all hyperparameters — visible in the MLflow dashboard
        mlflow.log_params(params)
        mlflow.log_param("batchsize", batchsize)
 
        optimizer = optim.Adam
        loss_fn = torch.nn.CrossEntropyLoss()
 
        # Unpack params — Conv1dGRU uses all four hyperopt params
        model = Conv1dGRU(**params)
        model.to(device)
 
        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optimizer,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )
        trainer.loop()
 
        # Save model with timestamp
        tag = datetime.now().strftime("%Y%m%d-%H%M")
        modelpath = modeldir / (tag + "model.pt")
        logger.info(f"Saving model to {modelpath}")
        torch.save(model, modelpath)
 
        mlflow.log_artifact(local_path=str(modelpath), artifact_path="pytorch_models")
 
        # trainer.test_loss holds the last validation loss
        # Hyperopt minimises loss — lower is better
        return {"loss": trainer.test_loss, "status": STATUS_OK}

In [96]:
with mlflow.start_run():
    ...


In [97]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")  # or "file:./mlruns"
mlflow.set_experiment("gestures-hyperopt")

<Experiment: artifact_location=('file:///c:/Users/IqraRafiq/OneDrive - '
 'Mysolution/Desktop/LS/MADS-MachineLearning-course/notebooks/3_recurrent_networks/mlruns/2'), creation_time=1774817087509, experiment_id='2', last_update_time=1774817087509, lifecycle_stage='active', name='gestures-hyperopt', tags={}>

In [94]:
# ── Main ─────────────────────────────────────────────────────────
def main():
    setup_mlflow("gestures-hyperopt")
 
    search_space = {
        "hidden_size":  scope.int(hp.quniform("hidden_size",  32, 256, 32)),
        "num_layers":   scope.int(hp.quniform("num_layers",    1,   3,  1)),
        "conv_filters": scope.int(hp.quniform("conv_filters", 16,  64, 16)),
        "kernel_size":  scope.int(hp.quniform("kernel_size",   3,   7,  2)),
    }
 
    best_result = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,   # smarter than random — learns from previous trials
        max_evals=10,        # set to 50-100 for a real sweep
        trials=Trials(),
    )
 
    logger.info(f"Best result: {best_result}")
 
 
if __name__ == "__main__":
    main()

  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

2026-03-29 23:04:03.663 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures
100%|##########| 651/651 [00:14<00:00, 46.50it/s]
2026-03-29 23:05:20.100 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-230520
2026-03-29 23:05:20.101 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|##########| 100/100 [00:05<00:00, 16.99it/s]
2026-03-29 23:05:26.773 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.5971 test 2.1682 metric ['0.2037']
100%|##########| 100/100 [00:05<00:00, 18.00it/s]
2026-03-29 23:05:33.183 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 1.8399 test 1.3509 metric ['0.4575']
100%|##########| 100/100 [00:05<00:00, 18.01it/s]
2026-03-29 23:05:39.553 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 1.0896 test 0.9542 metric ['0.6069']
100%|##########| 100/100 [00:05

 10%|█         | 1/10 [02:51<25:47, 172.00s/trial, best loss: 0.06650868495926261]

2026-03-29 23:06:55.670 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures
100%|##########| 651/651 [00:00<00:00, 2023.26it/s]
2026-03-29 23:06:58.277 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-230658
2026-03-29 23:06:58.280 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|##########| 100/100 [00:10<00:00,  9.76it/s]
2026-03-29 23:07:09.839 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.6082 test 2.2705 metric ['0.1819']
100%|##########| 100/100 [00:09<00:00, 10.63it/s]
2026-03-29 23:07:20.543 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 1.9882 test 1.5613 metric ['0.4213']
100%|##########| 100/100 [00:08<00:00, 11.34it/s]
2026-03-29 23:07:30.595 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 1.1904 test 0.8753 metric ['0.6044']
100%|##########| 100/100 [00:

 20%|██        | 2/10 [05:28<21:41, 162.63s/trial, best loss: 0.04085919747594744]

...

2026-03-29 23:30:21.498 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures
100%|##########| 651/651 [00:00<00:00, 1307.44it/s]
2026-03-29 23:30:25.876 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-233025
2026-03-29 23:30:25.878 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|##########| 100/100 [00:08<00:00, 12.22it/s]
2026-03-29 23:30:35.136 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.6340 test 2.3294 metric ['0.1450']
100%|##########| 100/100 [00:07<00:00, 13.40it/s]
2026-03-29 23:30:43.617 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.2324 test 2.1208 metric ['0.2338']
100%|##########| 100/100 [00:07<00:00, 13.58it/s]
2026-03-29 23:30:52.019 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 1.8891 test 1.8039 metric ['0.3063']
100%|##########| 100/100 [00:

 90%|█████████ | 9/10 [29:26<03:15, 195.57s/trial, best loss: 0.03195243728812784]

2026-03-29 23:33:30.385 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures
100%|##########| 651/651 [00:00<00:00, 1303.73it/s]
2026-03-29 23:33:33.397 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-233333
2026-03-29 23:33:33.402 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|##########| 100/100 [00:09<00:00, 10.73it/s]
2026-03-29 23:33:44.156 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.6414 test 2.2580 metric ['0.1850']
100%|##########| 100/100 [00:09<00:00, 10.62it/s]
2026-03-29 23:33:55.019 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.0842 test 1.6155 metric ['0.3944']
100%|##########| 100/100 [00:09<00:00, 10.63it/s]
2026-03-29 23:34:05.951 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 1.4841 test 1.1236 metric ['0.5056']
100%|##########| 100/100 [00:

100%|██████████| 10/10 [33:18<00:00, 199.87s/trial, best loss: 0.03195243728812784]

2026-03-29 23:37:22.387 | INFO     | __main__:main:20 - Best result: {'conv_filters': np.float64(64.0), 'hidden_size': np.float64(192.0), 'kernel_size': np.float64(4.0), 'num_layers': np.float64(1.0)}


In [65]:
def run_experiment(model: nn.Module, model_name: str, dev_name: str, extra_params: dict = None):
    """
    Train a model and log everything to MLflow.
    Recreates streamers each run to avoid generator exhaustion.
    """
    # Recreate streamers — generators exhaust after one epoch
    s = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
    tr = s["train"].stream()
    va = s["valid"].stream()
 
    with mlflow.start_run():
        mlflow.set_tag("model", model_name)
        mlflow.set_tag("dev", dev_name)
 
        params = {
            "model_name": model_name,
            "n_params": sum(p.numel() for p in model.parameters()),
        }
        if extra_params:
            params.update(extra_params)
        mlflow.log_params(params)
 
        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=tr,
            validdataloader=va,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )
        trainer.loop()
        # Save model
        
        modelpath = modeldir / ( model_name + ".pt")
        torch.save(model, modelpath)
        print(f"Saved to {modelpath}")

In [67]:
# --- Experiment 1: Baseline GRU ---
print("=== Experiment 1: Baseline GRU (hidden=64, layers=1, dropout=0) ===")
cfg1 = GRUConfig(hidden_size=64, num_layers=1, dropout=0.0)
run_experiment(
    GRUModel(cfg1), "GRU-baseline", "your-name",
    {"hidden_size": 64, "num_layers": 1, "dropout": 0.0}
)

2026-03-29 22:29:40.837 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures


=== Experiment 1: Baseline GRU (hidden=64, layers=1, dropout=0) ===


2026-03-29 22:29:41.698 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-222941
2026-03-29 22:29:41.698 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:02<00:00, 33.61it/s]
2026-03-29 22:29:44.337 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.8424 test 2.4870 metric ['0.1187']
100%|██████████| 81/81 [00:02<00:00, 36.03it/s]
2026-03-29 22:29:46.786 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.3257 test 2.2268 metric ['0.1844']
100%|██████████| 81/81 [00:02<00:00, 34.93it/s]
2026-03-29 22:29:49.254 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 2.1433 test 2.0808 metric ['0.2719']
100%|██████████| 3/3 [00:07<00:00,  2.52s/it]

Saved to C:\Users\IqraRafiq\OneDrive - Mysolution\Desktop\LS\MADS-MachineLearning-course\notebooks\3_recurrent_networks\gestures\GRU-baseline.pt


In [68]:
# --- Experiment 2: GRU with dropout ---
print("=== Experiment 2: GRU + dropout=0.3 ===")
cfg2 = GRUConfig(hidden_size=64, num_layers=1, dropout=0.3)
run_experiment(
    GRUWithDropout(cfg2), "GRU-dropout", "your-name",
    {"hidden_size": 64, "num_layers": 1, "dropout": 0.3}
)

2026-03-29 22:29:49.298 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures


=== Experiment 2: GRU + dropout=0.3 ===


2026-03-29 22:29:49.681 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-222949
2026-03-29 22:29:49.681 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:01<00:00, 42.70it/s]
2026-03-29 22:29:51.740 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.8877 test 2.5631 metric ['0.1047']
100%|██████████| 81/81 [00:02<00:00, 38.27it/s]
2026-03-29 22:29:54.020 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.3789 test 2.2880 metric ['0.1891']
100%|██████████| 81/81 [00:01<00:00, 41.19it/s]
2026-03-29 22:29:56.137 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 2.1844 test 2.0312 metric ['0.2953']
100%|██████████| 3/3 [00:06<00:00,  2.15s/it]

Saved to C:\Users\IqraRafiq\OneDrive - Mysolution\Desktop\LS\MADS-MachineLearning-course\notebooks\3_recurrent_networks\gestures\GRU-dropout.pt


In [69]:
# --- Experiment 3: Deeper GRU ---
print("=== Experiment 3: GRU 2 layers, hidden=128 ===")
cfg3 = GRUConfig(hidden_size=128, num_layers=2, dropout=0.3)
run_experiment(
    GRUWithDropout(cfg3), "GRU-deep", "your-name",
    {"hidden_size": 128, "num_layers": 2, "dropout": 0.3}
)

2026-03-29 22:29:56.184 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures


=== Experiment 3: GRU 2 layers, hidden=128 ===


2026-03-29 22:29:56.554 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-222956
2026-03-29 22:29:56.556 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:06<00:00, 13.32it/s]
2026-03-29 22:30:03.053 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.5937 test 2.2665 metric ['0.2141']
100%|██████████| 81/81 [00:06<00:00, 13.23it/s]
2026-03-29 22:30:09.573 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.0538 test 1.7468 metric ['0.3844']
100%|██████████| 81/81 [00:05<00:00, 13.99it/s]
2026-03-29 22:30:15.765 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 1.5066 test 1.1987 metric ['0.5500']
100%|██████████| 3/3 [00:19<00:00,  6.40s/it]

Saved to C:\Users\IqraRafiq\OneDrive - Mysolution\Desktop\LS\MADS-MachineLearning-course\notebooks\3_recurrent_networks\gestures\GRU-deep.pt


In [70]:
# --- Experiment 4: LSTM ---
print("=== Experiment 4: LSTM hidden=128, layers=2 ===")
cfg4 = GRUConfig(hidden_size=128, num_layers=2, dropout=0.3)
run_experiment(
    LSTMModel(cfg4), "LSTM", "your-name",
    {"hidden_size": 128, "num_layers": 2, "dropout": 0.3}
)

2026-03-29 22:30:15.822 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures


=== Experiment 4: LSTM hidden=128, layers=2 ===


2026-03-29 22:30:16.207 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-223016
2026-03-29 22:30:16.224 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:03<00:00, 23.95it/s]
2026-03-29 22:30:19.821 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.6228 test 2.2523 metric ['0.2359']
100%|██████████| 81/81 [00:03<00:00, 25.08it/s]
2026-03-29 22:30:23.251 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.0482 test 1.7831 metric ['0.2844']
100%|██████████| 81/81 [00:03<00:00, 24.54it/s]
2026-03-29 22:30:26.738 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 1.6967 test 1.6244 metric ['0.3594']
100%|██████████| 3/3 [00:10<00:00,  3.50s/it]


Saved to C:\Users\IqraRafiq\OneDrive - Mysolution\Desktop\LS\MADS-MachineLearning-course\notebooks\3_recurrent_networks\gestures\LSTM.pt


In [71]:
# --- Experiment 5: Conv1d + GRU (should reach >90%) ---
print("=== Experiment 5: Conv1dGRU — target >90% ===")
conv_cfg = ConvGRUConfig(
    conv_filters=32, kernel_size=3,
    hidden_size=128, num_layers=2, dropout=0.3
)
run_experiment(
    Conv1dGRU(conv_cfg), "Conv1dGRU", "your-name",
    {
        "conv_filters": conv_cfg.conv_filters,
        "kernel_size": conv_cfg.kernel_size,
        "hidden_size": conv_cfg.hidden_size,
        "num_layers": conv_cfg.num_layers,
        "dropout": conv_cfg.dropout,
    }
)

2026-03-29 22:30:26.794 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\IqraRafiq\.cache\mads_datasets\gestures


=== Experiment 5: Conv1dGRU — target >90% ===


2026-03-29 22:30:27.212 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260329-223027
2026-03-29 22:30:27.215 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:05<00:00, 13.88it/s]
2026-03-29 22:30:33.477 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.7089 test 2.3079 metric ['0.1750']
100%|██████████| 81/81 [00:05<00:00, 13.60it/s]
2026-03-29 22:30:39.860 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.2137 test 2.0443 metric ['0.2578']
100%|██████████| 81/81 [00:06<00:00, 13.23it/s]
2026-03-29 22:30:46.442 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 1.7922 test 1.4691 metric ['0.4203']
100%|██████████| 3/3 [00:19<00:00,  6.40s/it]

Saved to C:\Users\IqraRafiq\OneDrive - Mysolution\Desktop\LS\MADS-MachineLearning-course\notebooks\3_recurrent_networks\gestures\Conv1dGRU.pt
